# Visual retrieval complet : trouver des photographies avec CLIP

Cas pratique : un petit moteur de recherche dans cinq photos publiques. Télécharger la collection, indexer les embeddings CLIP, entrer une requête texte, afficher le top-k, lancer une recherche avec une nouvelle photo externe, puis exporter classement et index. Les résultats proviennent du calcul CLIP lors de l’exécution, jamais d’un classement préécrit.

Aucun entraînement. Les poids publics sont téléchargés au premier lancement. CPU possible, GPU facultatif. Cette démonstration complète du pipeline n’est pas un benchmark représentatif.

## 1. Dépendances

In [ ]:
%pip install "torch>=2.2,<3" "transformers>=4.45,<5" "pillow>=10,<13" "matplotlib>=3.8,<4"

In [ ]:
from pathlib import Path
import time, urllib.request
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
DATA=Path('data'); DATA.mkdir(exist_ok=True)
OUT=Path('outputs'); OUT.mkdir(exist_ok=True)
def download(url, name):
    path=DATA/name
    if not path.exists():
        with urllib.request.urlopen(url, timeout=60) as r:
            path.write_bytes(r.read())
    return path


## 2. Petite collection réelle
Images publiques de démonstration : chien PyTorch, bus et football Ultralytics. Les droits restent ceux des sources. Remplacer par vos propres images pour un projet réel.

In [ ]:
sources=[
('dog','https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg'),
('bus','https://ultralytics.com/images/bus.jpg'),
('football','https://ultralytics.com/images/zidane.jpg'),
('parrots','https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/hub/parrots.png'),
('cats','https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/coco_sample.png')]
images=[]
for name,url in sources:
    path=download(url,name+'.jpg')
    with Image.open(path) as im: images.append(im.convert('RGB'))
labels=[s[0] for s in sources]
fig,axes=plt.subplots(1,len(images),figsize=(16,4))
for ax,im,label in zip(axes,images,labels):ax.imshow(im);ax.set_title(label);ax.axis('off')
plt.suptitle('Collection réelle téléchargée : aucune image générée');plt.show()
assert len(images)==5


## 3. Charger CLIP et indexer les images une seule fois
L’index est la matrice normalisée des embeddings. Pour cinq images, un produit matriciel exhaustif suffit, sans serveur vectoriel. Les noms de fichiers ne participent pas au score CLIP.

In [ ]:
import torch, json, csv
from transformers import CLIPModel, CLIPProcessor
model_id='openai/clip-vit-base-patch32'
device='cuda' if torch.cuda.is_available() else 'cpu'
model=CLIPModel.from_pretrained(model_id).to(device).eval()
processor=CLIPProcessor.from_pretrained(model_id)
# Utiliser les sorties complètes CLIP pour garder une API stable.
with torch.inference_mode():
    batch=processor(images=images,text=['a photograph'],return_tensors='pt',padding=True).to(device)
    I=model(**batch).image_embeds
    I=I/I.norm(dim=-1,keepdim=True)
np.save(OUT/'clip_index.npy',I.cpu().numpy())
(OUT/'clip_manifest.json').write_text(json.dumps({'model':model_id,'items':[{'id':name,'url':url} for name,url in sources]},indent=2),encoding='utf-8')
print('Index :',tuple(I.shape),'| appareil :',device)


## 4. Une vraie requête texte, un classement calculé
Changer QUERY et relancer cette cellule. Le modèle a surtout appris l’anglais : commencer avec une requête anglaise. Le cosinus n’est pas une probabilité.

In [ ]:
def text_search(query,k=3):
    with torch.inference_mode():
        batch=processor(images=images[:1],text=[query],return_tensors='pt',padding=True).to(device)
        T=model(**batch).text_embeds
        T=T/T.norm(dim=-1,keepdim=True)
        scores=(T@I.T).cpu().numpy()[0]
    ids=np.argsort(-scores)[:min(k,len(images))]
    return ids,scores

def display_results(ids,scores,title):
    fig,axes=plt.subplots(1,len(ids),figsize=(4*len(ids),4),squeeze=False)
    for ax,j in zip(axes.ravel(),ids):
        ax.imshow(images[j]);ax.axis('off');ax.set_title(f'{labels[j]} | cos={float(scores[j]):.3f}')
    fig.suptitle(title);plt.show()

QUERY='a photo of a dog';K=3
t0=time.perf_counter();ids,scores=text_search(QUERY,K)
print('Requête :',QUERY,'| secondes :',round(time.perf_counter()-t0,3))
display_results(ids,scores,QUERY)
with (OUT/'recherche_texte.csv').open('w',encoding='utf-8',newline='') as f:
    w=csv.writer(f);w.writerow(['rang','image','cosinus','source'])
    for rank,j in enumerate(ids,1):w.writerow([rank,labels[j],float(scores[j]),sources[j][1]])
assert np.isfinite(scores).all()


## 5. Recherche image-image : une requête externe
La photo BLIP sert uniquement de requête : elle n’est pas présente dans la collection. Afficher la requête avant les résultats pour pouvoir juger leur pertinence. IMAGE_QUERY_PATH accepte aussi une photo locale.

In [ ]:
IMAGE_QUERY_PATH=None
query_path=Path(IMAGE_QUERY_PATH) if IMAGE_QUERY_PATH else download('https://storage.googleapis.com/sfr-vision-language-research/BLIP/demo.jpg','query_blip.jpg')
query_image=Image.open(query_path).convert('RGB')
plt.figure(figsize=(6,4));plt.imshow(query_image);plt.axis('off');plt.title('Photo requête externe, absente de la base');plt.show()
with torch.inference_mode():
    batch=processor(images=[query_image],text=['a photograph'],return_tensors='pt',padding=True).to(device)
    Q=model(**batch).image_embeds;Q=Q/Q.norm(dim=-1,keepdim=True)
    image_scores=(Q@I.T).cpu().numpy()[0]
ids_img=np.argsort(-image_scores)[:K]
display_results(ids_img,image_scores,'Photos les plus proches de la requête externe')
print([(labels[j],round(float(image_scores[j]),3)) for j in ids_img])


## 6. Évaluer plusieurs requêtes avec cibles déclarées
Protocole simplifié : exactement une image pertinente par requête. Hit@k équivaut ici au Recall@k à cible unique et AP vaut 1/rang de la cible. Les cibles sont des annotations pédagogiques manuelles ; la collection de cinq photos est trop petite pour conclure sur les performances réelles.

In [ ]:
evaluation=[('a photo of a dog','dog'),('a bus in a street','bus'),('people playing football','football'),('colorful parrots','parrots'),('cats resting on a couch','cats')]
positions=[];report=[]
for query,target in evaluation:
    ranking,s=text_search(query,len(images))
    target_idx=labels.index(target)
    position=int(np.flatnonzero(ranking==target_idx)[0])+1
    positions.append(position)
    report.append({'query':query,'target':target,'first_result':labels[ranking[0]],'target_rank':position})
    print(report[-1])
positions=np.array(positions)
for k in [1,3,5]:print(f'Hit@{k}: {(positions<=k).mean():.3f}')
print('mAP (cible unique) :',float((1/positions).mean()))
(OUT/'evaluation_retrieval.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
assert len(positions)==5 and np.all((positions>=1)&(positions<=5))


## 7. Ce qui a été réalisé
Téléchargement de cinq photos, indexation, recherche texte-image, recherche avec une photo externe, affichage des résultats et export des scores, de l’index et de l’évaluation. Pour un projet réel : plusieurs milliers d’images, plusieurs pertinents par requête, requêtes variées et index FAISS. Évaluer les erreurs au lieu de supposer que le premier résultat est toujours correct.

## Questions
1. Pourquoi indexer une seule fois ?
2. Pourquoi utiliser une image requête absente de la collection ?
3. Que signifie la similarité cosinus ?
4. Pourquoi le score sur cinq photos ne suffit-il pas ?

## Sources et crédits
- Modèle : https://huggingface.co/openai/clip-vit-base-patch32
- CLIP : https://arxiv.org/abs/2103.00020
- Photos : PyTorch Hub, Ultralytics, Hugging Face documentation-images, COCO 2017 et BLIP Salesforce ; URLs exactes dans les cellules. Les droits restent ceux de leurs auteurs.
- Recette CLIP : https://huggingface.co/docs/transformers/model_doc/clip
Les cellules sont à exécuter : aucune inférence nouvelle n’est prétendue préexécutée.